<a href="https://colab.research.google.com/github/leeknnn/viktoriiashkorubska-41/blob/master/lab_work_15/lab_work_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare

# Варіант 1: Київ (base_temp = 9.5, amplitude = 14)
np.random.seed(1)
base_temp = 9.5
amplitude = 14
city = "Київ"

def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"

rows = []
for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):
        seasonal = amplitude * np.cos((month - 7) / 12 * 2 * np.pi)
        noise = np.random.normal(0, 3.0)
        temp = round(base_temp + seasonal + noise, 1)
        diff = temp - base_temp
        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"
        rows.append({
            "місто": city, "рік": year, "місяць": month,
            "температура": temp, "сезон": season(month),
            "відхилення_від_норми": norm_cat,
        })

climate = pd.DataFrame(rows)

print("--- Завдання 1: Таблиця спряженості ---")
table_counts = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"])
display(table_counts)

table_norm = pd.crosstab(climate["сезон"], climate["відхилення_від_норми"], normalize="index")
print("\nНормована за рядками таблиця спряженості:")
display(table_norm)

print("--- Завдання 2: Критерій незалежності хі-квадрат ---")
chi2, pval, dof, expected = chi2_contingency(table_counts)
print(f"chi2 = {chi2:.4f}, p-value = {pval:.4e}, dof = {dof}")

expected_df = pd.DataFrame(expected, index=table_counts.index, columns=table_counts.columns)
print("\nОчікувані частоти (expected):")
display(expected_df.round(2))

print("--- Завдання 3: Перевірка умови застосовності (f_exp >= 5) ---")
min_exp = expected.min()
print(f"Мінімальна очікувана частота: {min_exp:.2f}")

if min_exp < 5:
    print("Увага: Є клітинки з очікуваною частотою < 5. Об'єднуємо категорії...")
    climate["відхилення_обєдноване"] = climate["відхилення_від_норми"].replace({"холодніше": "не тепліше", "звичайно": "не тепліше"})
    table_merged = pd.crosstab(climate["сезон"], climate["відхилення_обєдноване"])
    display(table_merged)
    chi2_m, pval_m, dof_m, exp_m = chi2_contingency(table_merged)
    print(f"Після об'єднання: chi2 = {chi2_m:.4f}, p-value = {pval_m:.4e}")

print("\n--- Завдання 4: Критерій узгодженості для сезонів (рівномірний) ---")
obs_seasons = climate["сезон"].value_counts().reindex(["зима", "весна", "літо", "осінь"])
n_total = len(climate)
exp_seasons = [n_total / 4] * 4

res_seasons = chisquare(f_obs=obs_seasons, f_exp=exp_seasons)
print(f"Статистика chisquare = {res_seasons.statistic:.4f}, p-value = {res_seasons.pvalue:.4f}")

print("\n--- Завдання 5: Власна заявлена пропорція для відхилення від норми ---")
obs_norm = climate["відхилення_від_норми"].value_counts().reindex(["холодніше", "звичайно", "тепліше"]).fillna(0)
exp_prop = [0.35 * n_total, 0.30 * n_total, 0.35 * n_total]

res_norm = chisquare(f_obs=obs_norm, f_exp=exp_prop)
print(f"Спостережувані частоти: {dict(obs_norm)}")
print(f"Очікувані частоти: {exp_prop}")
print(f"Статистика chisquare = {res_norm.statistic:.4f}, p-value = {res_norm.pvalue:.4f}")

## Відповіді на контрольні питання

1. **У чому полягає механіка статистики $\chi^2$?**
   - Статистика $\chi^2$ вимірює сумарну відхиленість фактично спостережуваних частот ($O$) від теоретично очікуваних ($E$). Відхилення підносяться до квадрата $(O - E)^2$, щоб уникнути взаємного погашення додатних та від'ємних різниць і надати більшої ваги великим розбіжностям, а ділення на очікувану частоту $E$ масштабує це відхилення відносно розміру самих клітинок.

2. **Як обчислюється очікувана частота клітинки і чому ця формула коректна під припущенням незалежності?**
   - Очікувана частота клітинки обчислюється за формулою $E = \frac{\text{Сума рядка} \times \text{Сума стовпця}}{\text{Загальна сума}}$. Якщо дві змінні є статистично незалежними, то сумісна ймовірність їх одночасної появи дорівнює добутку їх крайових (маргінальних) ймовірностей, тому ця формула точно відображає кількість спостережень, яка має потрапити у клітинку за відсутності зв'язку між змінними.

3. **Чи означає «p-value велике», що доведено незалежність змінних?**
   - Ні, велике $p$-value ($p > \alpha$) означає лише те, що наявні дані не дають достатньо статистично значущих підстав відхилити нульову гіпотезу про незалежність. Вона приймається як робоча модель, але це не є абсолютним математичним доведенням суворої незалежності, оскільки для виявлення слабких зв'язків може просто не вистачати обсягу вибірки.

4. **Що робити, якщо очікувана частота в якійсь клітинці менша за 5, і чому це проблема?**
   - Коли очікувані частоти малі ($E < 5$), математичне апроксимування дискретного біноміального/мультиноміального розподілу неперервним розподілом хі-квадрат стає неточнішим і дає спотворені значення $p$-value. Для усунення цієї проблеми застосовують об'єднання суміжних/логічно споріднених категорій або використовують точний критерій Фішера.